# 🧱 Lab: Bug Log Regression Suite

**Module: Tests That Catch Real Bugs** | **Type: Wall Lab**

---

## Learning Objectives

By the end of this lab, you will be able to:

1. **Analyze** real bug scenarios from an AI application and identify the deterministic code that failed
2. **Build** a regression test suite that reproduces each bug using fixtures and test doubles
3. **Apply** the fake-client pattern to test parsing failures, retry exhaustion, and edge cases without network access
4. **Create** golden-file comparisons that catch output drift introduced by future refactors

## Concepts Covered

| Concept | Gloss | From |
|---------|-------|------|
| Deterministic vs Stochastic Behavior | Test what your code does with outputs you control, and measure model quality separately. Treat model sampling as stochastic, isolate it behind a boundary so unit tests stay reliable. | mini-testability-map |
| Unit Test | A small assertion about one behavior. A unit test that cannot fail is theatre, it should catch a real bug when the behavior breaks. | mini-first-pytest-tests |
| Test Fixture | Reusable sample data and helpers (like pytest fixtures and tmp_path) that make tests repeatable, isolated, and easy to read, often using golden-output comparison to catch regressions. | mini-fixtures-and-golden-file |
| Test Double (Fake AI Client) | An injected fake backend that can return malformed JSON or time out on demand, so tests reliably exercise error paths without calling real external services. | mini-fake-ai-client-tests |
| Golden-File Comparison | — | mini-fixtures-and-golden-file |
| Dependency Injection | Pass the backend in, callers choose which model client to use, while your pipeline code stays unchanged and testable. | mini-fake-ai-client-tests |
| Bounded Retry | Retry transient errors only, and cap attempts, total wall-clock time, and cost or rate-limit budget so failures end fast, predictably, and safely. | mini-fake-ai-client-tests |
| Output Parsing | Output parsing is turning a model’s response into reliable, structured data (like JSON fields) your code can safely use,while detecting mistakes, missing info, or prompt-injection tricks in the output. | mini-fake-ai-client-tests |

## Prerequisites

- **mini-testability-map**: Classifying pipeline stages as deterministic or stochastic
- **mini-first-pytest-tests**: Writing pytest assertions and the red-then-green discipline
- **mini-fixtures-and-golden-file**: Reusable fixtures, `tmp_path`, and golden-file comparisons
- **mini-fake-ai-client-tests**: Injecting a fake AI client to test retries, errors, and parsing offline


## The Scenario

You've been running an AI-powered **invoice extraction service** for a few weeks. It takes raw invoice text, sends it to an LLM, and parses structured data out of the response. Along the way, you kept a **bug log** — notes on every failure you hit in production.

Your bug log has five entries:

| Bug # | What Happened | Root Cause |
|-------|--------------|------------|
| 1 | Returned `None` total when invoice had a currency symbol | `parse_invoice()` didn't strip `$` before converting to float |
| 2 | Crash on multi-line LLM response with markdown fences | `extract_json_block()` couldn't find JSON inside ```json ... ``` fences |
| 3 | Silent data loss — missing fields returned as empty dict | `validate_invoice()` accepted dicts with missing required keys |
| 4 | App hung for 90 seconds then crashed | No retry limit on API timeouts |
| 5 | Report file changed subtly after refactor, nobody noticed | No golden-file test on `format_invoice_report()` |

Your goal: **turn each bug into a regression test** so it can never come back.

Every one of these bugs lives in **deterministic code** — parsing, validation, formatting, retry logic — not in the LLM's stochastic output. That means `assert ==` works perfectly.

## Setup

In [1]:
!pip install -q pytest ipytest


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\molak\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import ipytest
ipytest.autoconfig()

print("ipytest ready.")

ipytest ready.


## The Production Code

Below is the invoice extraction pipeline. Each function is a deterministic stage — exactly the kind of code you identified as testable in the testability map mini-lab.

In [3]:
import json
import re
import time
from pathlib import Path


def parse_invoice_total(raw_total: str) -> float:
    """Convert a raw total string like '$1,234.56' to a float."""
    cleaned = raw_total.replace(",", "").strip()
    # Strip common currency symbols
    cleaned = re.sub(r"^[£$€¥]", "", cleaned).strip()
    return float(cleaned)


def extract_json_block(llm_output: str) -> dict:
    """Extract JSON from LLM output that may be wrapped in markdown fences."""
    # Try to find ```json ... ``` fences first
    fence_match = re.search(r"```(?:json)?\s*\n?(.*?)\n?```", llm_output, re.DOTALL)
    if fence_match:
        return json.loads(fence_match.group(1).strip())
    # Fall back to raw JSON parsing
    return json.loads(llm_output.strip())


REQUIRED_FIELDS = {"vendor", "total", "date"}

def validate_invoice(data: dict) -> dict:
    """Validate that all required fields are present and non-empty."""
    missing = REQUIRED_FIELDS - set(data.keys())
    if missing:
        raise ValueError(f"Missing required fields: {sorted(missing)}")
    empty = [k for k in REQUIRED_FIELDS if not str(data[k]).strip()]
    if empty:
        raise ValueError(f"Empty required fields: {sorted(empty)}")
    return data


def format_invoice_report(invoices: list[dict], output_path: Path) -> Path:
    """Write a formatted invoice report to a file."""
    lines = ["# Invoice Report", ""]
    for inv in invoices:
        lines.append(f"- {inv['vendor']}: ${inv['total']:.2f} on {inv['date']}")
    lines.append("")
    lines.append(f"Total invoices: {len(invoices)}")
    output_path.write_text("\n".join(lines))
    return output_path


def extract_invoice(text: str, client, max_retries: int = 2) -> dict:
    """Full pipeline: send text to AI client, parse, validate."""
    messages = [
        {"role": "system", "content": "Extract vendor, total, and date as JSON."},
        {"role": "user", "content": text},
    ]
    last_error = None
    for attempt in range(max_retries + 1):
        try:
            response = client.chat.completions.create(
                model="gpt-4o-mini", messages=messages
            )
            raw = response.choices[0].message.content
            data = extract_json_block(raw)
            return validate_invoice(data)
        except (ValueError, json.JSONDecodeError) as e:
            raise ValueError(f"Parse/validation error: {e}") from e
        except Exception as e:
            last_error = e
            if attempt < max_retries:
                time.sleep(0)  # no real delay in tests
    raise RuntimeError(
        f"All {max_retries + 1} attempts failed. Last error: {last_error}"
    )


print("Invoice extraction pipeline defined — 5 functions to test.")

Invoice extraction pipeline defined — 5 functions to test.


## The Fake AI Client

Same pattern from the mini-lab — a test double that mimics the OpenAI client interface.

In [4]:
class FakeMessage:
    def __init__(self, content: str):
        self.content = content

class FakeChoice:
    def __init__(self, content: str):
        self.message = FakeMessage(content)

class FakeResponse:
    def __init__(self, content: str):
        self.choices = [FakeChoice(content)]


class FakeAIClient:
    """Test double for an OpenAI-style client."""

    def __init__(self, responses=None, error=None):
        self.responses = list(responses) if responses else []
        self.error = error
        self.call_count = 0
        self.chat = self
        self.completions = self

    def create(self, **kwargs):
        self.call_count += 1
        if self.error:
            raise self.error
        return FakeResponse(self.responses.pop(0))


print("FakeAIClient ready.")

FakeAIClient ready.


## Bug #1 — Currency Symbol Crash

**What happened:** `parse_invoice_total("$1,234.56")` returned `None` or crashed because the `$` wasn't stripped before float conversion.

**The regression test:** Verify that currency symbols are handled correctly.

In [5]:
%%ipytest -v

import pytest


def test_bug1_dollar_sign_stripped():
    """Bug #1: '$1,234.56' must parse to 1234.56."""
    assert parse_invoice_total("$1,234.56") == 1234.56


def test_bug1_euro_sign_stripped():
    """Bug #1 variant: euro symbol."""
    assert parse_invoice_total("€500.00") == 500.00


def test_bug1_plain_number_still_works():
    """Regression guard: plain numbers shouldn't break."""
    assert parse_invoice_total("99.99") == 99.99


def test_bug1_whitespace_around_total():
    """Edge case: whitespace around the value."""
    assert parse_invoice_total("  $50.00  ") == 50.00

======================================= test session starts =======================================
platform win32 -- Python 3.12.10, pytest-9.1.1, pluggy-1.6.0
rootdir: C:\Users\molak\MJ_Projects\19_atomic_ai_eng_course\1_ai_engineering_track\output\content\labs\lab-bug-log-regression-suite
plugins: anyio-4.9.0, asyncio-1.4.0
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 4 items

t_e8447ebfd0ec456d8ef6c57ce285bfaf.py 

.

.

.

.

                                                   [100%]



======================================== 4 passed in 0.10s ========================================


## Bug #2 — Markdown Fence JSON

**What happened:** The LLM wrapped its JSON in ` ```json ... ``` ` fences, and `json.loads()` choked on the backticks.

**The regression test:** Verify that `extract_json_block` handles both fenced and raw JSON.

In [6]:
%%ipytest -v


@pytest.fixture
def sample_invoice_dict():
    """The expected parsed result — reused across Bug #2 tests."""
    return {"vendor": "Acme Corp", "total": "$150.00", "date": "2025-01-15"}


def test_bug2_fenced_json(sample_invoice_dict):
    """Bug #2: JSON wrapped in ```json fences must parse."""
    fenced = '```json\n{"vendor": "Acme Corp", "total": "$150.00", "date": "2025-01-15"}\n```'
    assert extract_json_block(fenced) == sample_invoice_dict


def test_bug2_fenced_without_json_tag(sample_invoice_dict):
    """Bug #2 variant: fences without the 'json' tag."""
    fenced = '```\n{"vendor": "Acme Corp", "total": "$150.00", "date": "2025-01-15"}\n```'
    assert extract_json_block(fenced) == sample_invoice_dict


def test_bug2_raw_json_still_works(sample_invoice_dict):
    """Regression guard: raw JSON without fences still parses."""
    raw = '{"vendor": "Acme Corp", "total": "$150.00", "date": "2025-01-15"}'
    assert extract_json_block(raw) == sample_invoice_dict


def test_bug2_total_garbage_raises():
    """Non-JSON, non-fenced text raises an error."""
    with pytest.raises(json.JSONDecodeError):
        extract_json_block("I couldn't process that invoice.")

======================================= test session starts =======================================
platform win32 -- Python 3.12.10, pytest-9.1.1, pluggy-1.6.0
rootdir: C:\Users\molak\MJ_Projects\19_atomic_ai_eng_course\1_ai_engineering_track\output\content\labs\lab-bug-log-regression-suite
plugins: anyio-4.9.0, asyncio-1.4.0
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 4 items

t_e8447ebfd0ec456d8ef6c57ce285bfaf.py 

.

.

.

.

                                                   [100%]



======================================== 4 passed in 0.06s ========================================


## Bug #3 — Silent Data Loss (Missing Fields)

**What happened:** The LLM returned `{"vendor": "Acme"}` (missing `total` and `date`), and the code happily returned an incomplete dict. Downstream code crashed later with a confusing `KeyError`.

**The regression test:** Verify that `validate_invoice` rejects dicts with missing or empty required fields.

In [7]:
%%ipytest -v


def test_bug3_missing_fields_rejected():
    """Bug #3: dict missing 'total' and 'date' must raise ValueError."""
    with pytest.raises(ValueError, match="Missing required fields"):
        validate_invoice({"vendor": "Acme"})


def test_bug3_empty_field_rejected():
    """Bug #3 variant: field present but empty string."""
    with pytest.raises(ValueError, match="Empty required fields"):
        validate_invoice({"vendor": "Acme", "total": "", "date": "2025-01-15"})


def test_bug3_valid_dict_passes():
    """Regression guard: a complete dict passes validation."""
    data = {"vendor": "Acme", "total": "150.00", "date": "2025-01-15"}
    assert validate_invoice(data) == data

======================================= test session starts =======================================
platform win32 -- Python 3.12.10, pytest-9.1.1, pluggy-1.6.0
rootdir: C:\Users\molak\MJ_Projects\19_atomic_ai_eng_course\1_ai_engineering_track\output\content\labs\lab-bug-log-regression-suite
plugins: anyio-4.9.0, asyncio-1.4.0
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 3 items

t_e8447ebfd0ec456d8ef6c57ce285bfaf.py 

.

.

.

                                                    [100%]



======================================== 3 passed in 0.06s ========================================


## Bug #4 — Unbounded Retries on Timeout

**What happened:** The API timed out, and the retry loop had no upper bound — the app hung for 90 seconds before finally crashing.

**The regression test:** Use a fake client that always times out and verify that `extract_invoice` gives up after the configured number of retries.

In [8]:
%%ipytest -v


def test_bug4_retries_are_bounded():
    """Bug #4: API timeout with max_retries=2 → exactly 3 attempts, then RuntimeError."""
    fake = FakeAIClient(error=TimeoutError("Connection timed out"))

    with pytest.raises(RuntimeError, match="All 3 attempts failed"):
        extract_invoice("invoice text", client=fake, max_retries=2)

    assert fake.call_count == 3  # 1 initial + 2 retries


def test_bug4_zero_retries_means_one_attempt():
    """Bug #4 edge case: max_retries=0 → single attempt, no retries."""
    fake = FakeAIClient(error=ConnectionError("Network down"))

    with pytest.raises(RuntimeError, match="All 1 attempts failed"):
        extract_invoice("invoice text", client=fake, max_retries=0)

    assert fake.call_count == 1

======================================= test session starts =======================================
platform win32 -- Python 3.12.10, pytest-9.1.1, pluggy-1.6.0
rootdir: C:\Users\molak\MJ_Projects\19_atomic_ai_eng_course\1_ai_engineering_track\output\content\labs\lab-bug-log-regression-suite
plugins: anyio-4.9.0, asyncio-1.4.0
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 2 items

t_e8447ebfd0ec456d8ef6c57ce285bfaf.py 

.

.

                                                     [100%]



======================================== 2 passed in 0.04s ========================================


## Bug #5 — Output Drift in Report Formatting

**What happened:** Someone refactored `format_invoice_report` and accidentally changed the date format in the output. Nobody noticed until a customer complained about their reports looking different.

**The regression test:** A golden-file comparison that catches any change to the report output — even a single character.

In [9]:
%%ipytest -v


GOLDEN_INVOICE_REPORT = """# Invoice Report

- Acme Corp: $1234.56 on 2025-01-15
- Globex Inc: $789.00 on 2025-02-20

Total invoices: 2"""


@pytest.fixture
def invoices():
    """Two sample invoices — reusable fixture."""
    return [
        {"vendor": "Acme Corp", "total": 1234.56, "date": "2025-01-15"},
        {"vendor": "Globex Inc", "total": 789.00, "date": "2025-02-20"},
    ]


@pytest.fixture
def golden_path(tmp_path):
    """Write the golden file into the test's temp dir."""
    gf = tmp_path / "golden_report.md"
    gf.write_text(GOLDEN_INVOICE_REPORT)
    return gf


def test_bug5_report_matches_golden(invoices, golden_path, tmp_path):
    """Bug #5: report output must match golden file exactly."""
    actual_path = tmp_path / "actual_report.md"
    format_invoice_report(invoices, actual_path)

    actual = actual_path.read_text()
    expected = golden_path.read_text()

    assert actual == expected, (
        f"Output drift detected!\n"
        f"--- Expected ---\n{expected}\n"
        f"--- Actual ---\n{actual}"
    )

======================================= test session starts =======================================
platform win32 -- Python 3.12.10, pytest-9.1.1, pluggy-1.6.0
rootdir: C:\Users\molak\MJ_Projects\19_atomic_ai_eng_course\1_ai_engineering_track\output\content\labs\lab-bug-log-regression-suite
plugins: anyio-4.9.0, asyncio-1.4.0
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 1 item

t_e8447ebfd0ec456d8ef6c57ce285bfaf.py 

.

                                                      [100%]



======================================== 1 passed in 0.11s ========================================


## Full Integration: Fake Client → Parse → Validate

Now let's tie the bugs together by testing the **full `extract_invoice` pipeline** end-to-end — using the fake client to control the LLM response and verifying that parsing, fence-stripping, and validation all work as a chain.

In [10]:
%%ipytest -v


def test_full_pipeline_happy_path():
    """Fake client returns fenced JSON → parse → validate → correct dict."""
    fenced_response = (
        '```json\n'
        '{"vendor": "Acme Corp", "total": "$1,234.56", "date": "2025-01-15"}\n'
        '```'
    )
    fake = FakeAIClient(responses=[fenced_response])
    result = extract_invoice("Invoice from Acme Corp...", client=fake)

    assert result["vendor"] == "Acme Corp"
    assert result["date"] == "2025-01-15"
    assert fake.call_count == 1


def test_full_pipeline_malformed_json():
    """Fake client returns garbage → pipeline raises ValueError."""
    fake = FakeAIClient(responses=["I cannot process this invoice."])

    with pytest.raises(ValueError, match="Parse/validation error"):
        extract_invoice("bad invoice", client=fake)


def test_full_pipeline_missing_fields():
    """Fake client returns valid JSON but missing required fields."""
    fake = FakeAIClient(responses=['{"vendor": "Acme"}'])

    with pytest.raises(ValueError, match="Missing required fields"):
        extract_invoice("partial invoice", client=fake)

======================================= test session starts =======================================
platform win32 -- Python 3.12.10, pytest-9.1.1, pluggy-1.6.0
rootdir: C:\Users\molak\MJ_Projects\19_atomic_ai_eng_course\1_ai_engineering_track\output\content\labs\lab-bug-log-regression-suite
plugins: anyio-4.9.0, asyncio-1.4.0
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 3 items

t_e8447ebfd0ec456d8ef6c57ce285bfaf.py 

.

.

.

                                                    [100%]



======================================== 3 passed in 0.04s ========================================


## Run the Full Regression Suite

Let's run every test at once to see the full bug log as a green suite. Each test name maps back to the original bug number.

In [11]:
%%ipytest -v

import pytest


# ── Shared fixtures ────────────────────────────────────────

@pytest.fixture
def sample_invoice_dict():
    return {"vendor": "Acme Corp", "total": "$150.00", "date": "2025-01-15"}

@pytest.fixture
def invoices():
    return [
        {"vendor": "Acme Corp", "total": 1234.56, "date": "2025-01-15"},
        {"vendor": "Globex Inc", "total": 789.00, "date": "2025-02-20"},
    ]

@pytest.fixture
def golden_path(tmp_path):
    gf = tmp_path / "golden_report.md"
    gf.write_text(GOLDEN_INVOICE_REPORT)
    return gf


# ── Bug #1: Currency symbols ──────────────────────────────

def test_bug1_dollar_sign():
    assert parse_invoice_total("$1,234.56") == 1234.56

def test_bug1_euro_sign():
    assert parse_invoice_total("€500.00") == 500.00


# ── Bug #2: Markdown fences ───────────────────────────────

def test_bug2_fenced_json(sample_invoice_dict):
    fenced = '```json\n{"vendor": "Acme Corp", "total": "$150.00", "date": "2025-01-15"}\n```'
    assert extract_json_block(fenced) == sample_invoice_dict

def test_bug2_raw_json(sample_invoice_dict):
    raw = '{"vendor": "Acme Corp", "total": "$150.00", "date": "2025-01-15"}'
    assert extract_json_block(raw) == sample_invoice_dict


# ── Bug #3: Missing/empty fields ──────────────────────────

def test_bug3_missing_fields():
    with pytest.raises(ValueError, match="Missing required fields"):
        validate_invoice({"vendor": "Acme"})

def test_bug3_empty_field():
    with pytest.raises(ValueError, match="Empty required fields"):
        validate_invoice({"vendor": "Acme", "total": "", "date": "2025-01-15"})


# ── Bug #4: Bounded retries ──────────────────────────────

def test_bug4_retries_bounded():
    fake = FakeAIClient(error=TimeoutError("timeout"))
    with pytest.raises(RuntimeError, match="All 3 attempts failed"):
        extract_invoice("text", client=fake, max_retries=2)
    assert fake.call_count == 3


# ── Bug #5: Golden-file report ───────────────────────────

def test_bug5_golden_report(invoices, golden_path, tmp_path):
    actual_path = tmp_path / "actual_report.md"
    format_invoice_report(invoices, actual_path)
    assert actual_path.read_text() == golden_path.read_text()


# ── Integration ──────────────────────────────────────────

def test_integration_happy_path():
    resp = '{"vendor": "Acme", "total": "100", "date": "2025-03-01"}'
    fake = FakeAIClient(responses=[resp])
    result = extract_invoice("invoice", client=fake)
    assert result["vendor"] == "Acme"
    assert fake.call_count == 1

======================================= test session starts =======================================
platform win32 -- Python 3.12.10, pytest-9.1.1, pluggy-1.6.0
rootdir: C:\Users\molak\MJ_Projects\19_atomic_ai_eng_course\1_ai_engineering_track\output\content\labs\lab-bug-log-regression-suite
plugins: anyio-4.9.0, asyncio-1.4.0
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 9 items

t_e8447ebfd0ec456d8ef6c57ce285bfaf.py 

.

.

.

.

.

... (2 outputs left out)

.

.

                                              [100%]


======================================== 9 passed in 0.18s ========================================


Every bug in the log is now a green test. If any of these bugs ever resurface — whether from a refactor, a library upgrade, or a copy-paste mistake — the test suite will catch it instantly, without needing an API key or network access.

## The Bug-to-Test Pattern

For any new bug you encounter:

| Step | Action |
|------|--------|
| 1 | **Identify the deterministic code** that failed (parsing, validation, formatting, retries) |
| 2 | **Reproduce the exact input** that triggered the bug |
| 3 | **Write a test** that fails with the buggy code and passes with the fix |
| 4 | **Use a fixture** if the setup is reused by multiple tests |
| 5 | **Use a fake client** if the bug involves the AI call path (retries, malformed responses) |
| 6 | **Use a golden file** if the bug involves output format drift |

## 🎯 Summary

### What You Built

You turned a five-entry bug log from a production AI invoice extraction service into a complete, green, network-free regression suite — each test pinning a specific bug so it can never silently return.

### Key Takeaways

1. **Every production bug is a test waiting to be written** — the bug gives you the exact input and the expected behavior, which is everything a test needs
2. **Test doubles make AI-path bugs testable offline** — fake clients that return malformed JSON or raise timeouts let you test retry and parsing logic without network access or cost
3. **Fixtures reduce duplication across related bug tests** — shared sample data and golden files are defined once and injected by name into every test that needs them
4. **Golden-file comparisons catch output drift** — a character-level comparison against a known-good snapshot detects formatting regressions that no individual `assert` would catch
5. **The full suite runs in milliseconds** — because every test targets deterministic glue code and uses fake dependencies, the entire regression suite is fast, free, and reliable